In [3]:
"""
VIX Volatility Sampler (Simple)
================================
Downloads historical VIX data via yfinance and samples random volatilities
by picking random indices from the historical series.
"""
"""
Generate Monte Carlo Volatility Simulation CSV
================================================
Columns: simulation, day, volatility
- 100,000 simulations × 1,260 days (5 years × 252 trading days)
- Volatility sampled from historical VIX using VIXVolatilitySampler

Output: volatility_simulations.csv
"""

import numpy as np
import pandas as pd
import yfinance as yf
import time


# ------------------------------------------------------------------ #
#  VIXVolatilitySampler (user's code)
# ------------------------------------------------------------------ #
class VIXVolatilitySampler:
    def __init__(self, period: str = "10y"):
        data = yf.download("^VIX", period=period, auto_adjust=True, progress=False)

        if data.empty:
            raise RuntimeError("Failed to download VIX data.")

        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        self.vix_close = data["Close"].dropna().values.flatten()
        self.vix_decimal = self.vix_close / 100.0
        self.n_obs = len(self.vix_decimal)

        print(f"Loaded {self.n_obs} VIX observations")
        print(f"VIX range: {self.vix_close.min():.2f}% – {self.vix_close.max():.2f}%")

    def sample(self, n: int = 1) -> np.ndarray:
        random_indices = np.random.randint(0, self.n_obs, size=n)
        return self.vix_decimal[random_indices]

In [4]:
# ------------------------------------------------------------------ #
#  Generate simulation data
# ------------------------------------------------------------------ #
if __name__ == "__main__":
    N_SIMULATIONS = 100_000
    N_DAYS = 5 * 252  # 1,260
    TOTAL_ROWS = N_SIMULATIONS * N_DAYS  # 126,000,000
    OUTPUT_FILE = "volatility_simulations.csv"

    # 1. Initialize sampler
    sampler = VIXVolatilitySampler(period="10y")

    # 2. Sample all volatilities at once
    print(f"\nGenerating {TOTAL_ROWS:,} rows ({N_SIMULATIONS:,} sims × {N_DAYS:,} days)...")
    start = time.time()

    all_vols = sampler.sample(n=TOTAL_ROWS)

    # 3. Build columns
    simulations = np.repeat(np.arange(N_SIMULATIONS), N_DAYS)
    days = np.tile(np.arange(1, N_DAYS + 1), N_SIMULATIONS)

    # 4. Save
    df = pd.DataFrame({
        "simulation": simulations,
        "day": days,
        "volatility": all_vols
    })

    print(f"Saving to {OUTPUT_FILE}...")
    df.to_csv(OUTPUT_FILE, index=False)

    elapsed = time.time() - start
    print(f"\nDone! {elapsed:.1f}s")
    print(f"Shape: {df.shape}")
    print(df.head(10))

Loaded 2515 VIX observations
VIX range: 9.14% – 82.69%

Generating 126,000,000 rows (100,000 sims × 1,260 days)...
Saving to volatility_simulations.csv...

Done! 309.3s
Shape: (126000000, 3)
   simulation  day  volatility
0           0    1      0.1616
1           0    2      0.1604
2           0    3      0.1897
3           0    4      0.2312
4           0    5      0.1711
5           0    6      0.1319
6           0    7      0.1363
7           0    8      0.1565
8           0    9      0.1741
9           0   10      0.2228
